# Module 8: Comparative District Ranking Index

**Task 8**: Normalize results and develop a comparative index to rank districts based on intensity of change (urban growth rate, % water body loss, decrease in vegetation).

**Deliverables**:
- Raw indicator table
- Normalized indicator table
- Composite index & ranking table
- Choropleth maps (2 states)
- Radar charts (top-5 districts)
- Ranked bar charts

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from config import (
    STATES, YEARS, LULC_CLASSES,
    PROCESSED_DIR, FIGURES_DIR, BOUNDARIES_DIR, GEE_EXPORTS_DIR,
)
from scripts.ranking import (
    compute_indicators, min_max_normalize, compute_composite_index,
)
from scripts.visualization import (
    save_figure, plot_ranking_bar, plot_radar_chart, plot_district_choropleth,
)
from scripts.lulc_utils import save_composition_csv

print('Imports successful!')

## 8.1: Load District Compositions for 2016 and 2025

In [ ]:
district_comps = {}  # {state_key: {year: DataFrame}}

for state_key, state_cfg in STATES.items():
    district_comps[state_key] = {}
    
    for year in [2016, 2025]:
        filepath = GEE_EXPORTS_DIR / f'{state_key}_district_composition_{year}.csv'
        if filepath.exists():
            df = pd.read_csv(filepath)
            # Ensure 'district' column exists
            if 'district' not in df.columns and 'ADM2_NAME' in df.columns:
                df = df.rename(columns={'ADM2_NAME': 'district'})
            district_comps[state_key][year] = df
            print(f'Loaded: {state_cfg["name"]} {year} ({len(df)} districts)')
        else:
            print(f'⚠️ Not found: {filepath.name}')

## 8.2: Compute Indicators

In [ ]:
all_indicators = {}

for state_key, state_cfg in STATES.items():
    if 2016 not in district_comps.get(state_key, {}) or \
       2025 not in district_comps.get(state_key, {}):
        print(f'⚠️ Missing data for {state_cfg["name"]}')
        continue
    
    indicators = compute_indicators(
        district_comps[state_key][2016],
        district_comps[state_key][2025],
        2016, 2025
    )
    all_indicators[state_key] = indicators
    
    print(f'\n{state_cfg["name"]} — Raw Indicators:')
    display(indicators[['district', 'urban_growth_rate', 'water_loss_pct', 'veg_decrease_pct']])
    
    save_composition_csv(
        indicators,
        PROCESSED_DIR / 'index' / f'{state_key}_raw_indicators.csv'
    )

## 8.3: Normalize & Compute Composite Index

In [ ]:
all_rankings = {}

for state_key, state_cfg in STATES.items():
    if state_key not in all_indicators:
        continue
    
    # Equal-weight composite index
    ranking = compute_composite_index(all_indicators[state_key])
    all_rankings[state_key] = ranking
    
    print(f'\n{state_cfg["name"]} — District Rankings:')
    display(ranking[['rank', 'district', 'norm_urban_growth_rate',
                      'norm_water_loss_pct', 'norm_veg_decrease_pct',
                      'composite_index']])
    
    save_composition_csv(
        ranking,
        PROCESSED_DIR / 'index' / f'{state_key}_composite_ranking.csv'
    )
    
    # Also compute with weighted version
    weighted = compute_composite_index(
        all_indicators[state_key],
        weights={'urban_growth_rate': 0.4, 'water_loss_pct': 0.3, 'veg_decrease_pct': 0.3}
    )
    save_composition_csv(
        weighted,
        PROCESSED_DIR / 'index' / f'{state_key}_weighted_ranking.csv'
    )

## 8.4: Visualization

In [ ]:
# Ranked bar charts
for state_key, state_cfg in STATES.items():
    if state_key not in all_rankings:
        continue
    plot_ranking_bar(all_rankings[state_key], state_cfg['name'])

print('✅ Ranking bar charts generated.')

In [ ]:
# Radar charts for top-5 districts
for state_key, state_cfg in STATES.items():
    if state_key not in all_rankings:
        continue
    n = min(5, len(all_rankings[state_key]))
    plot_radar_chart(all_rankings[state_key], state_cfg['name'], top_n=n)

print('✅ Radar charts generated.')

In [ ]:
# Choropleth maps
for state_key, state_cfg in STATES.items():
    if state_key not in all_rankings:
        continue
    
    boundary_file = BOUNDARIES_DIR / f'{state_key}_district_boundaries.geojson'
    if not boundary_file.exists():
        continue
    
    gdf = gpd.read_file(boundary_file)
    ranking = all_rankings[state_key]
    
    # Merge
    merge_col = 'ADM2_NAME' if 'ADM2_NAME' in gdf.columns else gdf.columns[0]
    merged = gdf.merge(ranking, left_on=merge_col, right_on='district', how='left')
    
    plot_district_choropleth(
        merged, 'composite_index',
        f'Composite Change Index — {state_cfg["name"]} (2016→2025)',
        cmap='RdYlGn_r',
        legend_title='Change Index',
        state_name=state_key
    )

print('\n✅ Module 8 complete.')

---
## Summary
- ✅ Three indicators computed per district: urban growth, water loss, vegetation decrease
- ✅ Min-max normalization (0–1)
- ✅ Equal-weight and weighted composite indices
- ✅ District rankings for both states
- ✅ Bar charts, radar charts, choropleth maps

**Next**: `09_validation.ipynb`